# Visualisation 2 — Regional Name Effects in France

**Research questions:**
- Is there a regional effect in naming data?
- Are some names more popular in specific departments?
- Are nationally popular names uniformly popular across France?

**Dashboard:**
- **Map 1** — Each department coloured by $η$: how much the local #1 name beats the national #1 name locally. High $η$ = strong regional naming identity.
- **Map 2** — Click any department on Map 1 to compare its naming culture to every other department ($η_{k,S,R}$ metric, $k=3$ top names).
- **Decade dropdown** — Re-renders both maps for the selected decade.

> GeoJSON geometry is embedded inline (simplified to 2 d.p., metro France only) — no network fetch required at render time.

## 1 — Imports

In [1]:
import json
import requests
from pathlib import Path

import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

GEO_URL = (
    'https://raw.githubusercontent.com/gregoiredavid/'
    'france-geojson/master/departements-avec-outre-mer.geojson'
)

## 2 — Load and filter data

In [2]:
df = pd.read_csv(
    '../dpt2020.csv',
    sep=';',
    dtype={'dpt': str, 'annais': str, 'sexe': int, 'nombre': int},
)

# Remove national aggregates (XX), year aggregates (XXXX), and rare-name bucket
df = df[
    (df['dpt'] != 'XX') &
    (df['annais'] != 'XXXX') &
    (df['preusuel'] != '_PRENOMS_RARES')
].copy()

df['annais'] = df['annais'].astype(int)
df['decade'] = (df['annais'] // 10) * 10

print(f"Rows after filter: {len(df):,}")
print(f"Departments: {df['dpt'].nunique()}")
print(f"Decades: {sorted(df['decade'].unique())}")

Rows after filter: 3,668,274
Departments: 99
Decades: [np.int64(1900), np.int64(1910), np.int64(1920), np.int64(1930), np.int64(1940), np.int64(1950), np.int64(1960), np.int64(1970), np.int64(1980), np.int64(1990), np.int64(2000), np.int64(2010), np.int64(2020)]


## 3 — Aggregate by (decade, dpt, preusuel)

In [3]:
agg = (
    df.groupby(['decade', 'dpt', 'preusuel'], sort=False)['nombre']
    .sum()
    .reset_index()
)

agg_gender = (
    df.groupby(['decade', 'dpt', 'preusuel', 'sexe'], sort=False)['nombre']
    .sum()
    .reset_index()
)

print(f"Aggregated rows: {len(agg):,}")

Aggregated rows: 745,155


## 4 — Compute η per (decade, dpt) for Map 1

$$\eta_{dpt} = \frac{|\text{local top name}|_{dpt}}{|\text{national top name}|_{dpt}} \geq 1$$

$\eta = 1$: the local #1 name IS the national #1 name.  
$\eta > 1$: a different name is locally more popular than the national #1.

In [4]:
K_DISPLAY  = 5  # names shown in Map 1 hover tooltip
K_PAIRWISE = 3  # names used in Map 2 pairwise η

eta_rows = []
for decade, d in agg.groupby('decade'):
    global_top = d.groupby('preusuel')['nombre'].sum().idxmax()

    local_top = (
        d.sort_values('nombre', ascending=False)
        .groupby('dpt', sort=False)
        .first()
        .reset_index()
        .rename(columns={'preusuel': 'top_name', 'nombre': 'top_count'})
    )

    global_in_dept = (
        d[d['preusuel'] == global_top][['dpt', 'nombre']]
        .rename(columns={'nombre': 'global_count'})
    )

    merged = local_top.merge(global_in_dept, on='dpt', how='left')
    merged['global_count'] = merged['global_count'].fillna(1)
    merged['eta'] = merged['top_count'] / merged['global_count']
    merged['decade'] = decade
    eta_rows.append(merged[['decade', 'dpt', 'eta', 'top_name']])

eta_long = pd.concat(eta_rows, ignore_index=True)
print(eta_long.shape)
eta_long.head()

(1257, 4)


,decade,dpt,eta,top_name
0,1900,29,1.0,MARIE
1,1900,57,1.0,MARIE
2,1900,75,1.0,MARIE
3,1900,59,1.0,MARIE
4,1900,56,1.0,MARIE


## 5 — Format top-k tooltip string per (decade, dpt)

In [5]:
gender_wide = agg_gender.pivot_table(
    index=['decade', 'dpt', 'preusuel'],
    columns='sexe',
    values='nombre',
    fill_value=0,
).reset_index()
gender_wide.columns.name = None

for col in [1, 2]:
    if col not in gender_wide.columns:
        gender_wide[col] = 0
gender_wide = gender_wide.rename(columns={1: 'count_m', 2: 'count_f'})
gender_wide['total'] = gender_wide['count_m'] + gender_wide['count_f']
gender_wide['pct_m'] = (
    (gender_wide['count_m'] / gender_wide['total'].replace(0, np.nan)) * 100
).round(0).fillna(0).astype(int)

agg_g = agg.merge(
    gender_wide[['decade', 'dpt', 'preusuel', 'pct_m']],
    on=['decade', 'dpt', 'preusuel'],
    how='left',
)


def get_topk_series(group: pd.DataFrame) -> pd.Series:
    top = group.nlargest(K_DISPLAY, 'nombre')
    res = {}
    
    for i, (_, r) in enumerate(top.iterrows(), 1):
        pct_m = int(r['pct_m'])
        if pct_m > 50:
            sex_per = f"🔵 {pct_m}% M"
        elif pct_m < 50:
            sex_per = f"🔴 {100-pct_m}% F"
        else:
            sex_per = f"⚪ 50-50% Mixte"
        
        res[f'top_{i}'] = f"{r['preusuel']} ({int(r['nombre']):,}, {sex_per})"
        
    for i in range(1, K_DISPLAY + 1):
        if f'top_{i}' not in res:
            res[f'top_{i}'] = "-"
            
    return pd.Series(res)


topk_split = (
    agg_g.groupby(['decade', 'dpt'], sort=False)
    .apply(get_topk_series, include_groups=False)
    .reset_index()
)

eta_long = eta_long.merge(topk_split, on=['decade', 'dpt'], how='left')
eta_long.head(3)

,decade,dpt,eta,top_name,top_1,top_2,top_3,top_4,top_5
0,1900,29,1.0,MARIE,"MARIE (27,115, 🔴 100% F)","JEAN (11,251, 🔵 100% M)","JEANNE (5,861, 🔴 100% F)","FRANÇOIS (4,359, 🔵 100% M)","ANNE (4,213, 🔴 100% F)"
1,1900,57,1.0,MARIE,"MARIE (14,906, 🔴 98% F)","JEAN (4,846, 🔵 100% M)","ANNE (4,499, 🔴 100% F)","JOSEPH (3,235, 🔵 100% M)","CATHERINE (2,455, 🔴 100% F)"
2,1900,75,1.0,MARIE,"MARIE (14,840, 🔴 98% F)","SUZANNE (14,743, 🔴 100% F)","ANDRÉ (13,264, 🔵 100% M)","GERMAINE (12,676, 🔴 100% F)","JEANNE (12,083, 🔴 100% F)"


## 6 — Compute pairwise $η_{k,S,R}$ for Map 2

$$\eta_{k,S,R} = \frac{\sum_{i=1}^{k}|\text{i-th top name of } R|_R}{\sum_{i=1}^{k}|\text{i-th top name of } S|_R} \geq 1$$

$η_{k,S,R} \simeq 1$ → $R$ has a similar naming culture to $S$.  
$η_{k,S,R} \gg 1$ → $R$'s own names are much more popular there than $S$'s top names → very different cultures.

*This cell takes ~30–60 s for 12 decades × 96 departments.*

In [6]:
pairwise_rows = []

for decade, d in agg.groupby('decade'):
    topk = (
        d.sort_values('nombre', ascending=False)
        .groupby('dpt', sort=False)
        .head(K_PAIRWISE)
    )
    topk_sums = topk.groupby('dpt')['nombre'].sum()

    pivot = d.pivot_table(
        index='preusuel',
        columns='dpt',
        values='nombre',
        fill_value=0,
        aggfunc='sum',
    )

    for S, s_group in topk.groupby('dpt'):
        s_names = s_group['preusuel'].tolist()
        s_in_r = pivot.loc[pivot.index.isin(s_names)].sum()
        eta_series = topk_sums / s_in_r.replace(0, np.nan)

        for R, val in eta_series.items():
            pairwise_rows.append({
                'decade': decade,
                'dpt':    S,
                'R_dpt':  R,
                'eta_kSR': val,
            })

pairwise_long = pd.DataFrame(pairwise_rows)
print(f"Pairwise rows: {len(pairwise_long):,}")
pairwise_long.head()

Pairwise rows: 121,623


,decade,dpt,R_dpt,eta_kSR
0,1900,01,01,1.000000
1,1900,01,02,1.147143
2,1900,01,03,1.000000
3,1900,01,04,1.106772
4,1900,01,05,1.070996


## 7 — Cache GeoJSON locally and build department name lookup

In [7]:
geo_path = Path('departements-avec-outre-mer.geojson')

if not geo_path.exists():
    print('Downloading GeoJSON …')
    r = requests.get(GEO_URL, timeout=30)
    r.raise_for_status()
    geo_path.write_bytes(r.content)
    print('Saved.')
else:
    print('GeoJSON already cached.')

with open(geo_path, encoding='utf-8') as f:
    geo_json = json.load(f)

# Lightweight name lookup (no geometry)
dept_names = pd.DataFrame([
    {'code': feat['properties']['code'], 'nom': feat['properties']['nom']}
    for feat in geo_json['features']
])

# Simplified metro features for inline embedding:
# • restrict to metropolitan France (code < '97') — removes overseas territories
# • round coordinates to 2 decimal places (~750 m precision) — ~70% size reduction
# Embedding inline eliminates the GitHub network fetch at every chart render.
def _round_coords(obj, prec=2):
    if isinstance(obj, float):
        return round(obj, prec)
    if isinstance(obj, list):
        return [_round_coords(x, prec) for x in obj]
    return obj

metro_features = [
    {
        'type': 'Feature',
        'properties': {'code': f['properties']['code'], 'nom': f['properties']['nom']},
        'geometry': {
            'type': f['geometry']['type'],
            'coordinates': _round_coords(f['geometry']['coordinates']),
        },
    }
    for f in geo_json['features']
    if f['properties']['code'] < '97'
]
metro_codes = {f['properties']['code'] for f in metro_features}

inline_kb = len(json.dumps(metro_features)) // 1024
print(f"Metro departments: {len(metro_features)}  |  Inline GeoJSON size: ~{inline_kb} KB")
dept_names.head()

GeoJSON already cached.
Metro departments: 96  |  Inline GeoJSON size: ~2649 KB


,code,nom
0,971,Guadeloupe
1,972,Martinique
2,973,Guyane
3,976,Mayotte
4,01,Ain


## 8 — Interactive Altair dashboard

**How to use:**
1. Choose a decade with the slider — both maps update.
2. Hover over any department on **Map 1** to see η and the top-5 names.
3. Click a department on **Map 1** → **Map 2** shows how culturally similar every other department is.
4. Click anywhere on the white background (outside the maps) to clear your department selection.

In [8]:
decades_list = sorted(eta_long['decade'].unique().tolist())
default_decade = 2000 if 2000 in decades_list else decades_list[len(decades_list) // 2]



top_cols = [f'top_{i}' for i in range(1, K_DISPLAY + 1)]

eta_wide = eta_long.pivot(index='dpt', columns='decade', values=['eta', 'top_name'] + top_cols)
eta_wide.columns = [f'{field}_{decade}' for field, decade in eta_wide.columns]
eta_wide = eta_wide.reset_index()
eta_wide = eta_wide[eta_wide['dpt'].isin(metro_codes)].reset_index(drop=True)
eta_wide_lookup_fields = [c for c in eta_wide.columns if c != 'dpt']

pair_wide = pairwise_long.pivot_table(index='R_dpt', columns=['decade', 'dpt'], values='eta_kSR')
pair_wide.columns = [f'{s}_{decade}' for decade, s in pair_wide.columns]
pair_wide = pair_wide.reset_index()

metro_s_cols = [c for c in pair_wide.columns if c != 'R_dpt' and c.rsplit('_', 1)[0] in metro_codes]
pair_wide = pair_wide[['R_dpt'] + metro_s_cols]
eta_kSR_cols = [c for c in pair_wide.columns if c != 'R_dpt']
pair_wide[eta_kSR_cols] = pair_wide[eta_kSR_cols].replace([np.inf, -np.inf], 10.0).fillna(1.0).clip(upper=10.0)

topnames_wide = eta_long.pivot(index='dpt', columns='decade', values=top_cols)
topnames_wide.columns = [f'{field}_{d}' for field, d in topnames_wide.columns]
topnames_wide = topnames_wide.reset_index().rename(columns={'dpt': 'R_dpt'})

pair_wide = pair_wide.merge(topnames_wide, on='R_dpt', how='left')
pair_wide = pair_wide[pair_wide['R_dpt'].isin(metro_codes)].reset_index(drop=True)
pair_wide_lookup_fields = [c for c in pair_wide.columns if c != 'R_dpt']



decade_param = alt.param(
    name='decade_param',
    value=default_decade,
    bind=alt.binding_range(
        min=min(decades_list), max=max(decades_list), step=10,
        name="Decade: "
    )
)
dept_sel = alt.selection_point(fields=['dpt'], name='dept_sel', empty='none')

tooltips_top_names = [alt.Tooltip(f'top_{i}:N', title=f"top {i}") for i in range(1, K_DISPLAY + 1)]


# ── Map 1: η choropleth ───────────────────────────────────────────────────
map1 = (
    alt.Chart(alt.InlineData(values=metro_features, format=alt.DataFormat(type="json")))
    .mark_geoshape(stroke='white', strokeWidth=0.5)
    .transform_calculate(dpt='datum.properties.code', nom='datum.properties.nom')
    .transform_lookup(lookup='dpt', from_=alt.LookupData(eta_wide, 'dpt', eta_wide_lookup_fields))
    .transform_calculate(
        eta="datum['eta_' + decade_param]",
        top_name="datum['top_name_' + decade_param]",
        **{f"top_{i}": f"datum['top_{i}_' + decade_param]" for i in range(1, K_DISPLAY + 1)}
    )
    .encode(
        color=alt.condition(
            dept_sel,
            alt.Color('eta:Q', scale=alt.Scale(scheme='yelloworangered', domainMin=1.0), title='η',
                      legend=alt.Legend(orient='bottom', gradientLength=180)),
            alt.value('#a8d8f0'),
        ),
        tooltip=[
            alt.Tooltip('nom:N', title='Department'),
            alt.Tooltip('eta:Q', title='η', format='.3f'),
            alt.Tooltip('top_name:N', title='Local #1'),
        ] + tooltips_top_names
    )
    .add_params(decade_param, dept_sel)
    .project('mercator')
    .properties(
        width=480, height=520,
        title=alt.TitleParams(
            text='η — Regional Name Distinctiveness',
            subtitle='Click a department to compare →'
        )
    )
)


# ── Map 2: η_{k,S,R} comparison ─────────────────────────────────────────────
map2_bg = (
    alt.Chart(alt.InlineData(values=metro_features, format=alt.DataFormat(type="json")))
    .mark_geoshape(fill='#e0e0e0', stroke='white', strokeWidth=0.5)
    .project('mercator')
    .properties(width=480, height=520)
)

map2_fg = (
    alt.Chart(alt.InlineData(values=metro_features, format=alt.DataFormat(type="json")))
    .mark_geoshape(stroke='white', strokeWidth=0.5)
    .transform_calculate(R_dpt='datum.properties.code', nom='datum.properties.nom')
    .transform_lookup(lookup='R_dpt', from_=alt.LookupData(pair_wide, 'R_dpt', pair_wide_lookup_fields))
    .transform_calculate(
        eta_kSR="isValid(dept_sel.dpt) ? datum[dept_sel.dpt + '_' + decade_param] : null",
        **{f"top_{i}": f"datum['top_{i}_' + decade_param]" for i in range(1, K_DISPLAY + 1)}
    )
    .transform_filter('isValid(datum.eta_kSR)')
    .encode(
        color=alt.Color('eta_kSR:Q', scale=alt.Scale(scheme='redyellowgreen', reverse=True, domainMin=1.0),
                         title='η_k,S,R', legend=alt.Legend(orient='bottom', gradientLength=180)),
        tooltip=[
            alt.Tooltip('nom:N', title='Department'),
            alt.Tooltip('eta_kSR:Q', title='η_k,S,R', format='.3f'),
        ] + tooltips_top_names
    )
    .project('mercator')
    .properties(width=480, height=520)
)

map2 = alt.layer(map2_bg, map2_fg).properties(
    title=alt.TitleParams(
        text='η_{k,S,R} — Naming Culture Similarity',
        subtitle=f'Green ≈ similar to selected dept  |  k = {K_PAIRWISE} names',
    )
)

# ── Layout final ──────────────────────────────────────────────────────────
dashboard = alt.hconcat(map1, map2).resolve_scale(color='independent')

dashboard

alt.HConcatChart(...)